# How the data infrastructure works

Code block 6 of [docs/data-infrastructure.md](https://github.com/quarcs-lab/gdo4bolivia/blob/main/docs/data-infrastructure.md) (GeoDevelopment Observatory of Bolivia), on its own. Run the cell (Runtime > Run all); it installs what it needs.

In [ ]:
!pip install -q geo4bolivia
import geo4bolivia as geo

pop = geo.load("pop", years=2020)

# up a nested hierarchy, with the metadata rules; n_units and coverage say how many units contributed
geo.aggregate(pop, to="department", dataset="pop")
geo.aggregate(pop, to=["province", "department", "country"], dataset="pop", variables=["pop"])

# SDG indicators: census-population-weighted means, the weights fetched from "sdg" automatically
mpi = geo.load("sdg01", variables="sdg1_2_mpi")
geo.aggregate(mpi, to="department", dataset="sdg01")

# your own rules and weights, e.g. poverty weighted by gridded population instead
geo.aggregate(geo.load("sdg01", variables="sdg1_2_mpi", years=2012), to="department",
              rules={"sdg1_2_mpi": "weighted_mean:pop"}, weights=geo.load("pop", years=2012))

# onto units that do not nest: a grid, watersheds, a newer municipality map
cells = geo.overlay(pop, my_grid, extensive=["pop"], id_col="cell")
weights = geo.overlay_weights("map343", my_zones, id_col="zone")   # the area shares used